In [1]:
# # TRS Article Data Extraction Pipeline
# Resumable pipeline: PDF -> TXT -> Mistral extraction -> output.xlsx + validation.xlsx
# Each `# %%` block is one notebook cell. Requires: pandas, openpyxl, mistralai, tqdm, pdftotext (poppler).
# STUDY = one row per study; GROUP = one row per group; OUTCOMES = one row per outcome (per template README).

In [2]:
# Cell 1: Packages and configuration
import os, json, subprocess
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from mistralai.client import Mistral

ROOT = Path("/home/paulocabral/Documents/AI_article_data_extraction")
PDF_DIR, TXT_DIR = ROOT / "pdfs_to_extract", ROOT / "pdfs_extracted"
GUIDE_DIR = ROOT / "guide_docs"
CONTROL_FILE = ROOT / "control_file.csv"
TEMPLATE = GUIDE_DIR / "TRS_data_extraction_template.xlsx"
VARS_FILE = GUIDE_DIR / "TRS_lista_de_variaveis_e_descricoes.xlsx"
RELEVANT_TABS = ["STUDY", "GROUP", "OUTCOMES"]
NFAS = "NFAS"

MISTRAL_API_KEY = "mstrl_M4JnVaCVSKLmd0VerknyMgYB2K4B47q9_1747HI" # Place your mistral API key here
MODEL = "mistral-large-latest"
client = Mistral(api_key=MISTRAL_API_KEY)

TXT_DIR.mkdir(exist_ok=True)


In [3]:
# Cell 2: PDF conversion
def pdf_to_text(pdf_path: Path, txt_path: Path) -> None:
    """Convert a PDF file to text using pdftotext, preserving layout."""
    subprocess.run(
        ["pdftotext", "-layout", str(pdf_path), str(txt_path)],
        check=True, capture_output=True, text=True,
    )


def ensure_txt(pdf_path: Path) -> Path:
    """Convert pdf if txt does not exist yet; return txt path."""
    txt_path = TXT_DIR / (pdf_path.stem + ".txt")
    if not txt_path.exists():
        pdf_to_text(pdf_path, txt_path)
    return txt_path


In [4]:
# Cell 3: Load template variables and their descriptions
def load_variable_definitions() -> dict:
    """Return {tab: {variable: description}} using the Aba/Variavel columns."""
    desc = pd.read_excel(VARS_FILE, sheet_name="Dicionario_de_variaveis")
    defs = {}
    for tab in RELEVANT_TABS:
        cols = [c for c in pd.read_excel(TEMPLATE, sheet_name=tab, nrows=0).columns]
        sub = desc[desc["Aba"] == tab]
        lookup = {
            str(v).strip(): str(d).strip()
            for v, d in zip(sub["Variável"], sub["Breve descrição"])
        }
        defs[tab] = {str(c).strip(): lookup.get(str(c).strip(), "") for c in cols}
    return defs


In [5]:
# Cell 4: Prompt construction
def build_prompt(tab: str, variables: dict, article_text: str) -> str:
    """Build the extraction prompt for one tab of one article."""
    var_lines = "\n".join(
        f'- "{name}": {desc or "no description available"}'
        for name, desc in variables.items()
        if name != "Study_ID"
    )
    n_expected = {"STUDY": "exactly ONE record", "GROUP": "one record per group",
                  "OUTCOMES": "one record per outcome/assessment"}
    return f"""You are a rigorous data-extraction assistant for a systematic review.
Tab to fill: {tab}. Return {n_expected[tab]}.

Variables and their descriptions:
{var_lines}

Article text (line numbers refer to this text, first line is 1):
<article>
{article_text}
</article>

Rules:
1. Extract values ONLY from the article text. Never invent or infer information.
2. If a variable is not found for a record, set its value to "NFAS" and lines to [].
3. "lines" are the 1-indexed line numbers of the article text where the value was found.
4. Do not include Study_ID; it is added automatically.

Answer ONLY with valid JSON, no markdown, following exactly this schema:
{{
  "records": [
    {{{{"<variable_name>": {{"value": "<extracted value or NFAS>", "lines": [<int>, ...]}}}}}}
  ]
}}"""


In [6]:
# Cell 5: Mistral call and response parsing
def parse_json(raw: str) -> dict:
    """Clean and parse the model response into a dict."""
    content = raw.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.lower().startswith("json"):
            content = content[4:]
    return json.loads(content)


def extract_tab(tab: str, variables: dict, article_text: str) -> list:
    """Call Mistral for one tab; return list of {variable: {value, lines}}."""
    response = client.chat.complete(
        model=MODEL,
        messages=[{"role": "user", "content": build_prompt(tab, variables, article_text)}],
    )
    records = parse_json(response.choices[0].message.content)["records"]
    result = []
    for record in records:
        row = {}
        for name in variables:
            item = record.get(name, {}) or {}
            value, lines = str(item.get("value", NFAS)), item.get("lines", [])
            if value.strip().upper() == NFAS:
                value, lines = NFAS, []
            row[name] = {"value": value, "lines": lines}
        result.append(row)
    return result


# %%
# Cell 6: Control file (resumability)
def init_control() -> pd.DataFrame:
    """Create or load the control file listing every article and its status."""
    pdfs = sorted(PDF_DIR.glob("*.pdf"))
    if CONTROL_FILE.exists():
        ctrl = pd.read_csv(CONTROL_FILE, dtype=str)
        known = set(ctrl["article"])
        new = [p.stem for p in pdfs if p.stem not in known]
        if new:
            ctrl = pd.concat([ctrl, pd.DataFrame({"article": new, "status": "pending"})],
                             ignore_index=True)
    else:
        ctrl = pd.DataFrame({"article": [p.stem for p in pdfs], "status": "pending"})
    return ctrl


def update_status(ctrl: pd.DataFrame, article: str, status: str) -> None:
    """Persist status for one article to disk."""
    ctrl.loc[ctrl["article"] == article, "status"] = status
    ctrl.to_csv(CONTROL_FILE, index=False)


# %%
# Cell 7: Output writers
def read_sheets(path: Path) -> dict:
    """Load all sheets of a workbook, or empty dict if it does not exist."""
    return pd.read_excel(path, sheet_name=None) if path.exists() else {}


def write_article(article: str, extraction: dict) -> None:
    """Append one article's results to output.xlsx and validation.xlsx."""
    out_path, val_path = ROOT / "output.xlsx", ROOT / "validation.xlsx"
    out_prev, val_prev = read_sheets(out_path), read_sheets(val_path)
    for tab, records in extraction.items():
        for record in records:
            record = {"Study_ID": article, **record}
            out_row = {v: d["value"] for v, d in record.items()}
            val_row = {"Study_ID": article}
            val_row.update({
                v: ", ".join(map(str, d["lines"])) if d["lines"] else NFAS
                for v, d in record.items() if v != "Study_ID"})
            out_prev[tab] = pd.concat(
                [out_prev.get(tab, pd.DataFrame()), pd.DataFrame([out_row])],
                ignore_index=True)
            val_prev[tab] = pd.concat(
                [val_prev.get(tab, pd.DataFrame()), pd.DataFrame([val_row])],
                ignore_index=True)
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for tab, df in out_prev.items():
            df.to_excel(writer, sheet_name=tab[:31], index=False)
    with pd.ExcelWriter(val_path, engine="openpyxl") as writer:
        for tab, df in val_prev.items():
            df.to_excel(writer, sheet_name=tab[:31], index=False)



In [7]:
# Cell 8: Main pipeline with progress bars
def process_article(article: str, defs: dict, ctrl: pd.DataFrame) -> None:
    """Convert, extract and persist one article; update control file."""
    update_status(ctrl, article, "converting")
    article_text = ensure_txt(PDF_DIR / f"{article}.pdf").read_text(
        encoding="utf-8", errors="replace")
    update_status(ctrl, article, "extracting")
    extraction = {}
    for tab in tqdm(RELEVANT_TABS, desc=f"{article} tabs", leave=False):
        extraction[tab] = extract_tab(tab, defs[tab], article_text)
    write_article(article, extraction)
    update_status(ctrl, article, "done")


def run_pipeline() -> None:
    """Execute the full resumable pipeline over all articles."""
    ctrl = init_control()
    defs = load_variable_definitions()
    pending = ctrl.loc[ctrl["status"] != "done", "article"].tolist()
    print(f"Total: {len(ctrl)} | Done: {(ctrl.status == 'done').sum()} | Pending: {len(pending)}")
    for article in tqdm(pending, desc="Articles"):
        try:
            process_article(article, defs, ctrl)
        except Exception as error:
            print(f"ERROR in {article}: {error}")
            update_status(ctrl, article, "error")

run_pipeline()

Total: 1 | Done: 0 | Pending: 1


Articles:   0%|          | 0/1 [00:00<?, ?it/s]

Capogrosso2013 tabs:   0%|          | 0/3 [00:00<?, ?it/s]